In [1]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time
import datetime
from urllib.request import urlopen, Request

In [2]:
r = Request('https://th.investing.com/economic-calendar/', headers={'User-Agent': 'Mozilla/5.0'})
time.sleep(2)
response = urlopen(r).read()
soup = BeautifulSoup(response, "html.parser")
table = soup.find_all(class_ = "js-event-item")
result = []

In [3]:
def event_calendar():
    
    for bl in table:
        event_datetime = bl.get('data-event-datetime', '').strip()
        currency = bl.find(class_="left flagCur noWrap").text.strip()
        intensity_divs = bl.find_all(class_="left textNum sentiment noWrap")
        event = bl.find(class_="left event").text.strip()
        intencity_val = 0
        true_count = 0

        for intence in intensity_divs:
            _true = intence.find_all(class_="grayFullBullishIcon")
            _false = intence.find_all(class_="grayEmptyBullishIcon")

            true_count = len(_true)

            if true_count == 3:
                intencity_val = 3
            elif true_count == 2:
                intencity_val = 2
            else :
                intencity_val = 1
                
        event_datetime = event_datetime.split(' ')
        date=event_datetime[0]
        time=event_datetime[1]
        
        
        result.append({'currency' : currency, 'date' : date, 'time' : time, 'intensity' : intencity_val, 'event':event})

    return result

In [4]:
news = event_calendar()
news_df = pd.DataFrame(news)

In [5]:
display(news_df.shape)
display(news_df.head(10))

(46, 5)

,currency,date,time,intensity,event
0,USD,2025/07/09,00:00:00,2,การประมูลธนบัตรอายุ 3 ปี
1,USD,2025/07/09,02:00:00,1,สินเชื่อผู้บริโภค ( พ.ค.)
2,USD,2025/07/09,03:30:00,2,รายงานสินค้าคงเหลือของน้ำมันดิบประจำไตรมาสจาก API
3,JPY,2025/07/09,06:50:00,1,ปริมาณเงิน M2 (ปีต่อปี)
4,JPY,2025/07/09,06:50:00,1,ปริมาณเงิน M3 (มิ.ย.)
5,AUD,2025/07/09,08:30:00,2,รายงานยอดการอนุมัติสินเชื่อเพื่อการสร้างบ้าน (...
6,AUD,2025/07/09,08:30:00,1,ยอดการอนุมัติโครงการสร้างบ้านภาคเอกชน ( พ.ค.)
7,AUD,2025/07/09,08:30:00,1,ภาพรวมแนวโน้มเศรษฐกิจและการเงิน (Chart Pack) โ...
8,CNY,2025/07/09,08:30:00,2,ดัชนีราคาผู้บริโภค (CPI) ของจีน (เดือนต่อเดือน...
9,CNY,2025/07/09,08:30:00,2,ดัชนีราคาผู้บริโภค(CPI) ของจีน (ปีต่อปี) (มิ.ย.)


In [6]:
now = datetime.datetime.now()
date_time = now.strftime("%Y-%m-%d %H-%M-%S").strip().replace(' ', '_')
news_df.to_csv(f'data/event_calendar/{date_time}.csv', index=False)

In [7]:
print(news_df[news_df['intensity'] == 3][['currency', 'event']])

   currency                              event
11      NZD  การตัดสินใจเกี่ยวกับอัตราดอกเบี้ย
33      USD              สินค้าคงคลังน้ำมันดิบ


In [8]:
import os
from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

load_dotenv("../sentiment_analysis/secret.env")

mongo_connection_string = os.getenv("MONGO_CONNECTION_STRING")

try:
    client = MongoClient(mongo_connection_string)
    db = client['stock_news_db']
    collection = db['event_calendar']

    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")

except Exception as e:
    print(f"An error occurred: {e}")

Pinged your deployment. You successfully connected to MongoDB!


In [9]:
complete_dict=news_df.to_dict(orient='records')

result = collection.insert_many(complete_dict,ordered=True)
print(f"Successfully inserted document with id: {result.inserted_ids}")

Successfully inserted document with id: [ObjectId('686e19d661b26aefe53bef2e'), ObjectId('686e19d661b26aefe53bef2f'), ObjectId('686e19d661b26aefe53bef30'), ObjectId('686e19d661b26aefe53bef31'), ObjectId('686e19d661b26aefe53bef32'), ObjectId('686e19d661b26aefe53bef33'), ObjectId('686e19d661b26aefe53bef34'), ObjectId('686e19d661b26aefe53bef35'), ObjectId('686e19d661b26aefe53bef36'), ObjectId('686e19d661b26aefe53bef37'), ObjectId('686e19d661b26aefe53bef38'), ObjectId('686e19d661b26aefe53bef39'), ObjectId('686e19d661b26aefe53bef3a'), ObjectId('686e19d661b26aefe53bef3b'), ObjectId('686e19d661b26aefe53bef3c'), ObjectId('686e19d661b26aefe53bef3d'), ObjectId('686e19d661b26aefe53bef3e'), ObjectId('686e19d661b26aefe53bef3f'), ObjectId('686e19d661b26aefe53bef40'), ObjectId('686e19d661b26aefe53bef41'), ObjectId('686e19d661b26aefe53bef42'), ObjectId('686e19d661b26aefe53bef43'), ObjectId('686e19d661b26aefe53bef44'), ObjectId('686e19d661b26aefe53bef45'), ObjectId('686e19d661b26aefe53bef46'), ObjectId(